# Pipeline Profiling Grid-Search

Visualizes results from `scripts/profiling_grid_search.py`.

**Grid:** OFFLINE × INSERT × BUFFER_REACT  ×  max_iterations ∈ {100, 500, 1000, 2000, 5000}

**Metrics tracked:**
- `driver_move_distance_km` — F.O (lower = better)
- `algo_elapsed_s` — algorithm wall time (inside `OptimizationSolver.solve()`)
- `total_s` — full cell wall time (solver + evaluate_solution)
- Stage breakdown: `fetch_s` (API mode only), `parse_s`, `validate_s`, `algo_elapsed_s`, `evaluate_s`

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DATA_DIR = Path('../data/profiling')

COLORS = {
    'OFFLINE':      '#3B82F6',   # blue
    'INSERT':       '#F97316',   # orange
    'BUFFER_REACT': '#10B981',   # green
}
SYMBOLS = {
    'OFFLINE':      'circle',
    'INSERT':       'square',
    'BUFFER_REACT': 'triangle-up',
}

LAYOUT_DEFAULTS = dict(
    template='plotly_white',
    font=dict(family='Inter, system-ui, sans-serif', size=13),
    margin=dict(l=60, r=30, t=60, b=60),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#e5e7eb',
        borderwidth=1,
    ),
    hoverlabel=dict(bgcolor='white', font_size=12),
)

## 1. Load Results

In [2]:
csv_files = sorted(DATA_DIR.glob('grid_search_*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No grid_search_*.csv found in {DATA_DIR}. Run scripts/profiling_grid_search.py first.')

latest = csv_files[-1]
print(f'Loading: {latest.name}')
df_raw = pd.read_csv(latest)

df_err = df_raw[df_raw['error'].notna()].copy()
df = df_raw[df_raw['error'].isna()].copy()

if not df_err.empty:
    print(f'⚠️  {len(df_err)} failed cells:')
    display(df_err[['algorithm', 'max_iter', 'error']])

df['max_iter'] = df['max_iter'].astype(int)

print(f'✓  {len(df)} successful cells | algorithms: {sorted(df["algorithm"].unique())} | max_iter values: {sorted(df["max_iter"].unique())}')
df.head()

Loading: grid_search_20260322_230314.csv
✓  15 successful cells | algorithms: ['BUFFER_REACT', 'INSERT', 'OFFLINE'] | max_iter values: [np.int64(100), np.int64(500), np.int64(1000), np.int64(2000), np.int64(5000)]


,algorithm,max_iter,driver_move_distance_km,total_labor_distance_km,vt_labors_assigned,services_successfully_assigned,services_total,utilization_without_moves_pct,algo_elapsed_s,solver_wall_s,solve_s,evaluate_s,total_s,run_id,error,fetch_s,parse_s,validate_s
0,OFFLINE,100,748.08,945.45,45,41,41,33.23,5.119262,5.130728,5.1308,0.0292,5.1601,fc2dab83-ede6-4f86-b16e-e38ba8199d8c,NaN,1.0606,0.2597,0.0029
1,OFFLINE,500,748.08,945.45,45,41,41,33.23,6.507548,6.524883,6.5250,0.0278,6.5529,cc2a9c7f-b0aa-488a-bf87-7390f8ac5f5c,NaN,1.0606,0.2597,0.0029
2,OFFLINE,1000,748.08,945.45,45,41,41,33.23,8.259044,8.284502,8.2846,0.0282,8.3130,fce1e7a9-ac53-4af9-8a60-7530afb0e232,NaN,1.0606,0.2597,0.0029
3,OFFLINE,2000,745.12,945.45,45,41,41,33.05,11.856772,11.900976,11.9011,0.0279,11.9292,a1e6ddcc-775f-4889-b9b9-e2775764075f,NaN,1.0606,0.2597,0.0029
4,OFFLINE,5000,745.12,945.45,45,41,41,33.05,21.113573,21.218633,21.2187,0.0283,21.2472,abc53f4a-cc8a-4392-b75d-47fa71f6c864,NaN,1.0606,0.2597,0.0029


## 2. Summary Table

In [3]:
algos = sorted(df['algorithm'].unique())
iters = sorted(df['max_iter'].unique())

metrics = [
    ('driver_move_distance_km',     'Driver Move Distance (km)',  'RdYlGn_r', '{:.2f}'),
    ('algo_elapsed_s',              'Algo Elapsed (s)',           'YlOrRd',   '{:.1f}'),
    ('services_successfully_assigned', 'Services Assigned',       'YlGn',     '{:.0f}'),
]

for col, title, cmap, fmt in metrics:
    if col not in df.columns:
        continue
    pivot = df.pivot_table(index='algorithm', columns='max_iter', values=col, aggfunc='first')
    pivot.index.name = 'Algorithm'
    pivot.columns.name = 'max_iter'

    # Build heatmap
    z = pivot.values.astype(float)
    fig = go.Figure(go.Heatmap(
        z=z,
        x=[str(c) for c in pivot.columns],
        y=list(pivot.index),
        text=[[fmt.format(v) for v in row] for row in z],
        texttemplate='%{text}',
        textfont=dict(size=14, family='monospace'),
        colorscale=cmap,
        showscale=True,
        colorbar=dict(thickness=14, len=0.8),
        hoverongaps=False,
        hovertemplate='Algorithm: %{y}<br>max_iter: %{x}<br>' + title + ': %{text}<extra></extra>',
    ))
    fig.update_layout(
        **LAYOUT_DEFAULTS,
        title=dict(text=f'<b>{title}</b>', font=dict(size=16)),
        xaxis=dict(title='max_iterations', tickfont=dict(size=12)),
        yaxis=dict(title='', tickfont=dict(size=13)),
        height=max(220, len(algos) * 80 + 120),
        width=700,
    )
    fig.show()

## 3. F.O vs max_iterations

In [10]:
fig = go.Figure()

for algo in sorted(df['algorithm'].unique()):
    grp = df[df['algorithm'] == algo].sort_values('max_iter')
    color = COLORS.get(algo, '#6B7280')
    symbol = SYMBOLS.get(algo, 'circle')

    hover = (
        '<b>' + algo + '</b><br>'
        'max_iter: %{x}<br>'
        'Driver Move Dist: <b>%{y:.2f} km</b><br>'
        'Services assigned: ' + grp['services_successfully_assigned'].astype(str) + '<br>'
        'Algo time: ' + grp['algo_elapsed_s'].round(1).astype(str) + 's'
        '<extra></extra>'
    )

    fig.add_trace(go.Scatter(
        x=grp['max_iter'],
        y=grp['driver_move_distance_km'],
        mode='lines+markers',
        name=algo,
        line=dict(color=color, width=2.5),
        marker=dict(symbol=symbol, size=10, color=color,
                    line=dict(color='white', width=1.5)),
        customdata=np.stack([
            grp['services_successfully_assigned'],
            grp['algo_elapsed_s'],
        ], axis=-1),
        hovertemplate=(
            '<b>' + algo + '</b><br>'
            'max_iter: %{x:,}<br>'
            'Driver Move Distance: <b>%{y:.2f} km</b><br>'
            'Services assigned: %{customdata[0]:.0f}<br>'
            'Algo time: %{customdata[1]:.1f}s'
            '<extra></extra>'
        ),
    ))

fig.update_layout(
    **LAYOUT_DEFAULTS,
    title=dict(text='<b>Objective Function vs max_iterations</b><br>'
               '<sup>driver_move_distance_km — lower is better</sup>',
               font=dict(size=16)),
    xaxis=dict(
        title='max_iterations',
        type='log',
        tickvals=sorted(df['max_iter'].unique()),
        ticktext=[f'{v:,}' for v in sorted(df['max_iter'].unique())],
        gridcolor='#f3f4f6',
    ),
    yaxis=dict(
        title='Driver Move Distance (km)',
        gridcolor='#f3f4f6',
    ),
    height=460,
    width=820,
)
fig.show()

## 4. Wall Time vs max_iterations

In [5]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Algorithm Time (algo_elapsed_s)', 'Total Cell Time (total_s)'),
    horizontal_spacing=0.12,
)

for algo in sorted(df['algorithm'].unique()):
    grp = df[df['algorithm'] == algo].sort_values('max_iter')
    color = COLORS.get(algo, '#6B7280')
    symbol = SYMBOLS.get(algo, 'circle')

    shared_kwargs = dict(
        name=algo,
        line=dict(color=color, width=2.5),
        marker=dict(symbol=symbol, size=10, color=color, line=dict(color='white', width=1.5)),
        legendgroup=algo,
    )

    fig.add_trace(go.Scatter(
        x=grp['max_iter'], y=grp['algo_elapsed_s'],
        mode='lines+markers',
        customdata=np.stack([grp['driver_move_distance_km'], grp['total_s']], axis=-1),
        hovertemplate=(
            f'<b>{algo}</b><br>'
            'max_iter: %{x:,}<br>'
            'Algo time: <b>%{y:.2f}s</b><br>'
            'F.O (move dist): %{customdata[0]:.2f} km<br>'
            'Total cell time: %{customdata[1]:.2f}s'
            '<extra></extra>'
        ),
        **shared_kwargs,
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=grp['max_iter'], y=grp['total_s'],
        mode='lines+markers',
        showlegend=False,
        customdata=np.stack([grp['algo_elapsed_s'], grp['evaluate_s']], axis=-1),
        hovertemplate=(
            f'<b>{algo}</b><br>'
            'max_iter: %{x:,}<br>'
            'Total time: <b>%{y:.2f}s</b><br>'
            'Algo: %{customdata[0]:.2f}s | Eval: %{customdata[1]:.3f}s'
            '<extra></extra>'
        ),
        **shared_kwargs,
    ), row=1, col=2)

tick_vals = sorted(df['max_iter'].unique())
tick_text = [f'{v:,}' for v in tick_vals]

for col_idx in [1, 2]:
    fig.update_xaxes(
        type='log', tickvals=tick_vals, ticktext=tick_text,
        title_text='max_iterations', gridcolor='#f3f4f6', row=1, col=col_idx,
    )
    fig.update_yaxes(title_text='Seconds', gridcolor='#f3f4f6', row=1, col=col_idx)

fig.update_layout(
    **LAYOUT_DEFAULTS,
    title=dict(text='<b>Wall Time vs max_iterations</b>', font=dict(size=16)),
    height=440,
    width=960,
)
fig.show()

## 5. Efficiency Frontier (F.O vs Compute Time)

In [6]:
fig = go.Figure()

for algo in sorted(df['algorithm'].unique()):
    grp = df[df['algorithm'] == algo].sort_values('max_iter')
    color = COLORS.get(algo, '#6B7280')
    symbol = SYMBOLS.get(algo, 'circle')

    # Connecting line (faded)
    fig.add_trace(go.Scatter(
        x=grp['algo_elapsed_s'], y=grp['driver_move_distance_km'],
        mode='lines',
        name=algo,
        legendgroup=algo,
        showlegend=False,
        line=dict(color=color, width=1, dash='dot'),
    ))

    # Points with max_iter labels
    fig.add_trace(go.Scatter(
        x=grp['algo_elapsed_s'],
        y=grp['driver_move_distance_km'],
        mode='markers+text',
        name=algo,
        legendgroup=algo,
        marker=dict(
            symbol=symbol, size=14, color=color,
            line=dict(color='white', width=2),
            opacity=0.95,
        ),
        text=[f'{v:,}' for v in grp['max_iter']],
        textposition='top right',
        textfont=dict(size=10, color=color),
        customdata=np.stack([
            grp['max_iter'],
            grp['driver_move_distance_km'],
            grp['total_s'],
            grp['services_successfully_assigned'],
        ], axis=-1),
        hovertemplate=(
            f'<b>{algo}</b><br>'
            'max_iter: %{customdata[0]:,.0f}<br>'
            'Algo time: <b>%{x:.2f}s</b><br>'
            'Driver Move Dist: <b>%{y:.2f} km</b><br>'
            'Total cell time: %{customdata[2]:.2f}s<br>'
            'Services assigned: %{customdata[3]:.0f}'
            '<extra></extra>'
        ),
    ))

fig.update_layout(
    **LAYOUT_DEFAULTS,
    title=dict(
        text='<b>Efficiency Frontier: Quality vs Compute</b><br>'
             '<sup>← left = faster &nbsp;&nbsp; ↓ down = better F.O &nbsp;&nbsp; labels = max_iterations</sup>',
        font=dict(size=16),
    ),
    xaxis=dict(title='Algorithm Time (seconds)  →  more compute', gridcolor='#f3f4f6'),
    yaxis=dict(title='Driver Move Distance (km)  ↓  lower is better', gridcolor='#f3f4f6'),
    height=520,
    width=820,
)
fig.show()

## 6. Stage Duration Breakdown

In [7]:
df_plot = df.copy().sort_values(['algorithm', 'max_iter'])
df_plot['label'] = df_plot['algorithm'] + ' / ' + df_plot['max_iter'].astype(str)
df_plot['overhead_s'] = (
    df_plot['total_s']
    - df_plot['algo_elapsed_s'].fillna(0)
    - df_plot['evaluate_s'].fillna(0)
).clip(lower=0)

stage_cols = [
    ('algo_elapsed_s', 'Algorithm iterations'),
    ('evaluate_s',     'evaluate_solution()'),
    ('overhead_s',     'Solver init / misc'),
]
stage_colors = ['#3B82F6', '#F97316', '#D1D5DB']

fig = go.Figure()

for (col, stage_label), color in zip(stage_cols, stage_colors):
    vals = df_plot[col].fillna(0)
    fig.add_trace(go.Bar(
        x=df_plot['label'],
        y=vals,
        name=stage_label,
        marker_color=color,
        hovertemplate=(
            '<b>%{x}</b><br>'
            f'{stage_label}: ' + '%{y:.3f}s'
            '<extra></extra>'
        ),
    ))

# Reference lines for one-time pipeline stages (constant across grid cells)
one_time = [
    ('fetch_s',    '#10B981', 'dash',   'API fetch'),
    ('parse_s',    '#EF4444', 'dot',    'parse input'),
    ('validate_s', '#8B5CF6', 'dashdot','validate'),
]
for col, color, dash, lbl in one_time:
    if col in df_plot.columns:
        val = df_plot[col].iloc[0]
        if pd.notna(val) and val > 0:
            fig.add_hline(
                y=val,
                line=dict(color=color, width=1.5, dash=dash),
                annotation_text=f'{lbl} ({val:.2f}s)',
                annotation_position='top right',
                annotation_font=dict(size=10, color=color),
            )

fig.update_layout(
    **LAYOUT_DEFAULTS,
    barmode='stack',
    title=dict(text='<b>Stage Duration Breakdown per Grid Cell</b>', font=dict(size=16)),
    xaxis=dict(title='Algorithm / max_iterations', tickangle=-35, tickfont=dict(size=10)),
    yaxis=dict(title='Wall time (seconds)', gridcolor='#f3f4f6'),
    height=500,
    width=max(820, len(df_plot) * 55),
)
fig.show()

# Stage share summary
print('\nStage share of total_s (mean across grid):')
for col, label in stage_cols:
    pct = (df_plot[col].fillna(0) / df_plot['total_s'] * 100).mean()
    bar = '█' * int(pct / 2)
    print(f'  {label:30s} {bar:25s} {pct:.1f}%')


Stage share of total_s (mean across grid):
  Algorithm iterations           █████████████████████████████████████████████████ 99.5%
  evaluate_solution()                                      0.3%
  Solver init / misc                                       0.2%


## 7. Marginal Gain Table (Diminishing Returns)

In [8]:
rows = []
for algo, grp in df.groupby('algorithm'):
    grp = grp.sort_values('max_iter').reset_index(drop=True)
    for i in range(1, len(grp)):
        prev, curr = grp.iloc[i - 1], grp.iloc[i]
        fo_prev, fo_curr = prev['driver_move_distance_km'], curr['driver_move_distance_km']
        delta_fo = fo_prev - fo_curr
        delta_fo_pct = (delta_fo / fo_prev * 100) if fo_prev > 0 else None
        delta_t = curr['algo_elapsed_s'] - prev['algo_elapsed_s']
        t_mult = curr['algo_elapsed_s'] / prev['algo_elapsed_s'] if prev['algo_elapsed_s'] > 0 else None
        is_knee = delta_fo_pct is not None and t_mult is not None and abs(delta_fo_pct) < 1.0 and t_mult > 2.0
        rows.append({
            'algorithm': algo,
            'step': f'{int(prev["max_iter"]):,} → {int(curr["max_iter"]):,}',
            'fo_prev_km': fo_prev,
            'fo_curr_km': fo_curr,
            'Δ F.O (km)': round(delta_fo, 3),
            'Δ F.O (%)': round(delta_fo_pct, 2) if delta_fo_pct is not None else None,
            'Δ time (s)': round(delta_t, 2),
            'time ×': round(t_mult, 2) if t_mult is not None else None,
            'knee': is_knee,
        })

df_m = pd.DataFrame(rows)

# --- Bar chart: Δ F.O % per step, coloured by algorithm ---
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('ΔF.O per step (%)', 'Δ algo time per step (×)'),
    horizontal_spacing=0.15,
)

for algo in sorted(df_m['algorithm'].unique()):
    sub = df_m[df_m['algorithm'] == algo]
    color = COLORS.get(algo, '#6B7280')

    fig.add_trace(go.Bar(
        x=sub['step'], y=sub['Δ F.O (%)'],
        name=algo, legendgroup=algo,
        marker_color=color,
        hovertemplate='<b>' + algo + '</b><br>Step: %{x}<br>Δ F.O: %{y:+.2f}%<extra></extra>',
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=sub['step'], y=sub['time ×'],
        name=algo, legendgroup=algo, showlegend=False,
        marker_color=color,
        hovertemplate='<b>' + algo + '</b><br>Step: %{x}<br>Time multiplier: %{y:.2f}×<extra></extra>',
    ), row=1, col=2)

# Knee threshold lines
fig.add_hline(y=1.0,  line=dict(color='#EF4444', width=1.5, dash='dash'),
              annotation_text='1% gain threshold', annotation_font_size=10,
              annotation_position='bottom right', row=1, col=1)
fig.add_hline(y=2.0,  line=dict(color='#EF4444', width=1.5, dash='dash'),
              annotation_text='2× time threshold', annotation_font_size=10,
              annotation_position='top right', row=1, col=2)

fig.update_layout(
    **LAYOUT_DEFAULTS,
    barmode='group',
    title=dict(text='<b>Marginal Gains — Diminishing Returns Analysis</b><br>'
               '<sup>Knee = <1% F.O gain but >2× time cost (highlighted in table below)</sup>',
               font=dict(size=16)),
    height=420,
    width=960,
)
fig.update_xaxes(tickangle=-30, tickfont=dict(size=10))
fig.update_yaxes(gridcolor='#f3f4f6')
fig.show()

# --- Styled table ---
def style_marginal(row):
    base = [''] * len(row)
    if row.get('knee'):
        return ['background-color: #fef3c7; color: #92400e'] * len(row)
    return base

display_cols = ['algorithm', 'step', 'fo_prev_km', 'fo_curr_km', 'Δ F.O (km)', 'Δ F.O (%)', 'Δ time (s)', 'time ×']
display(
    df_m[display_cols + ['knee']].style
    .apply(style_marginal, axis=1)
    .hide(axis='columns', subset=['knee'])
    .format({
        'fo_prev_km': '{:.2f}',
        'fo_curr_km': '{:.2f}',
        'Δ F.O (km)': '{:+.3f}',
        'Δ F.O (%)':  '{:+.2f}%',
        'Δ time (s)': '{:+.2f}s',
        'time ×':     '{:.2f}×',
    }, na_rep='—')
    .set_caption('⚠️ Yellow rows = diminishing returns zone (<1% F.O gain, >2× time cost)')
)

,algorithm,step,fo_prev_km,fo_curr_km,Δ F.O (km),Δ F.O (%),Δ time (s),time ×
0,BUFFER_REACT,100 → 500,748.08,748.08,+0.000,+0.00%,+1.17s,1.21×
1,BUFFER_REACT,"500 → 1,000",748.08,748.08,+0.000,+0.00%,+1.72s,1.25×
2,BUFFER_REACT,"1,000 → 2,000",748.08,745.12,+2.960,+0.40%,+3.36s,1.39×
3,BUFFER_REACT,"2,000 → 5,000",745.12,745.12,+0.000,+0.00%,+9.01s,1.76×
4,INSERT,100 → 500,95.79,125.47,-29.680,-30.98%,+12.09s,2.51×
5,INSERT,"500 → 1,000",125.47,107.90,+17.570,+14.00%,+17.15s,1.85×
6,INSERT,"1,000 → 2,000",107.90,112.44,-4.540,-4.21%,+32.61s,1.88×
7,INSERT,"2,000 → 5,000",112.44,101.64,+10.800,+9.61%,+101.44s,2.45×
8,OFFLINE,100 → 500,748.08,748.08,+0.000,+0.00%,+1.39s,1.27×
9,OFFLINE,"500 → 1,000",748.08,748.08,+0.000,+0.00%,+1.75s,1.27×


## 8. Timing Sanity Check

In [9]:
df_check = df.copy()
df_check['accounted_s'] = (
    df_check['fetch_s'].fillna(0)
    + df_check['algo_elapsed_s'].fillna(0)
    + df_check['evaluate_s'].fillna(0)
    + df_check['parse_s'].fillna(0)
    + df_check['validate_s'].fillna(0)
)
df_check['unaccounted_s'] = (df_check['total_s'] - df_check['accounted_s']).clip(lower=0)
df_check['unaccounted_pct'] = df_check['unaccounted_s'] / df_check['total_s'] * 100
df_check['label'] = df_check['algorithm'] + ' / ' + df_check['max_iter'].astype(str)

mean_pct = df_check['unaccounted_pct'].mean()
max_pct  = df_check['unaccounted_pct'].max()
status   = '✓ Good' if max_pct < 5 else '⚠️ Check stage accounting'
print(f'Timing sanity: mean unaccounted = {mean_pct:.1f}%  |  max = {max_pct:.1f}%  →  {status}')

# Waterfall-style stacked bar showing each stage's share of total_s
stage_display = [
    ('fetch_s',        'API fetch',         '#10B981'),
    ('parse_s',        'Parse input',       '#EF4444'),
    ('validate_s',     'Validate',          '#8B5CF6'),
    ('algo_elapsed_s', 'Algorithm',         '#3B82F6'),
    ('evaluate_s',     'Evaluate solution', '#F97316'),
    ('unaccounted_s',  'Unaccounted',       '#E5E7EB'),
]

fig = go.Figure()
for col, label, color in stage_display:
    if col not in df_check.columns:
        continue
    vals = df_check[col].fillna(0)
    pcts = vals / df_check['total_s'] * 100
    fig.add_trace(go.Bar(
        x=df_check['label'],
        y=pcts,
        name=label,
        marker_color=color,
        hovertemplate=(
            '<b>%{x}</b><br>'
            f'{label}: ' + '%{y:.1f}% of total_s'
            '<extra></extra>'
        ),
    ))

fig.update_layout(
    **LAYOUT_DEFAULTS,
    barmode='stack',
    title=dict(text='<b>Stage Share of Total Wall Time (%)</b><br>'
               '<sup>Unaccounted should be <5%</sup>', font=dict(size=16)),
    xaxis=dict(title='Algorithm / max_iterations', tickangle=-35, tickfont=dict(size=10)),
    yaxis=dict(title='% of total_s', gridcolor='#f3f4f6', range=[0, 105]),
    height=460,
    width=max(820, len(df_check) * 55),
)
fig.show()

# Table
cols = ['label', 'fetch_s', 'parse_s', 'validate_s', 'algo_elapsed_s', 'evaluate_s', 'total_s', 'unaccounted_s', 'unaccounted_pct']
cols = [c for c in cols if c in df_check.columns]
fmt_cols = [c for c in cols if c not in ('label', 'unaccounted_pct')]
display(
    df_check[cols].rename(columns={'label': 'cell'})
    .style
    .format('{:.3f}', subset=fmt_cols)
    .format('{:.1f}%', subset=['unaccounted_pct'])
    .background_gradient(cmap='RdYlGn_r', subset=['unaccounted_pct'], vmin=0, vmax=10)
)

Timing sanity: mean unaccounted = 0.0%  |  max = 0.0%  →  ✓ Good


,cell,fetch_s,parse_s,validate_s,algo_elapsed_s,evaluate_s,total_s,unaccounted_s,unaccounted_pct
0,OFFLINE / 100,1.061,0.260,0.003,5.119,0.029,5.160,0.000,0.0%
1,OFFLINE / 500,1.061,0.260,0.003,6.508,0.028,6.553,0.000,0.0%
2,OFFLINE / 1000,1.061,0.260,0.003,8.259,0.028,8.313,0.000,0.0%
3,OFFLINE / 2000,1.061,0.260,0.003,11.857,0.028,11.929,0.000,0.0%
4,OFFLINE / 5000,1.061,0.260,0.003,21.114,0.028,21.247,0.000,0.0%
5,INSERT / 100,1.061,0.260,0.003,8.029,0.024,8.066,0.000,0.0%
6,INSERT / 500,1.061,0.260,0.003,20.117,0.023,20.158,0.000,0.0%
7,INSERT / 1000,1.061,0.260,0.003,37.265,0.026,37.315,0.000,0.0%
8,INSERT / 2000,1.061,0.260,0.003,69.878,0.024,69.940,0.000,0.0%
9,INSERT / 5000,1.061,0.260,0.003,171.315,0.025,171.424,0.000,0.0%
